# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [11]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx")
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：..\data\E Commerce Dataset.xlsx
项目输出目录：c:\Users\Lenovo\Desktop\新建文件夹 (2)\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [12]:
# 在此写下你的答案：
#1.每条记录代表一个独立用户（客户）在平台上的行为与属性快照，以 CustomerID 唯一标识。它汇总了该用户的注册时长、登录设备偏好、支付方式、订单行为、满意度评分、投诉记录、返现金额等跨生命周期信息。
#2.Churn（流失标志）。取值为 1 表示用户已流失，0 表示用户仍留存。该列是后续预测建模或因果分析的核心因变量。
#3.语义上：CustomerID 只是用户的唯一标识符（如 50001、50002），其数值大小与任何业务属性无关，不存在"ID 越大用户越…"的物理意义。
#统计上：若将其当作连续数值输入模型，会引入虚假的序关系（如 50002 比 50001 "大"），导致模型学习到无意义的模式，甚至产生过拟合。
#实践上：它属于高基数标识列，应作为索引或用于去重/关联，而非作为特征进入回归、树模型或距离计算。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [13]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    report = pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": (data.isna().sum() / len(data) * 100).round(2),
        "唯一值数量": data.nunique()
    })
    report.index.name = "字段名"
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)   # Jupyter 环境
# print(quality_before) # 纯 Python 环境

,数据类型,缺失数量,缺失比例(%),唯一值数量
字段名,,,,
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,object,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,object,0,0.00,7
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [14]:
# TODO：完成项目初始审计
# 1. 完全重复行数（所有列都相同）
print("完全重复行数：", raw_df.duplicated().sum())

# 2. CustomerID 重复数量
# 注意：这里统计的是 CustomerID 出现重复的记录数（即重复次数，不含首次出现）
print("CustomerID 重复数量：", raw_df["CustomerID"].duplicated().sum())

# 3. Churn 频数与流失率
print("\nChurn 频数：")
print(raw_df["Churn"].value_counts())
print(f"流失率：{raw_df['Churn'].mean() * 100:.2f}%")

# 4. 主要类别字段的频数
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())


完全重复行数： 0
CustomerID 重复数量： 0

Churn 频数：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率：16.84%

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [15]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [16]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # 复制数据，避免覆盖原始数据
    df = data.copy()
    logs = []
    
    # 1. 删除完全重复行
    n_before = len(df)
    df = df.drop_duplicates()
    n_after = len(df)
    logs.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "删除所有列完全相同的重复记录，保留首次出现",
        "处理前记录数": n_before,
        "处理后记录数": n_after,
        "影响记录数": n_before - n_after
    })
    
    # 2. 数值型缺失值中位数填补（全局中位数，不使用 Churn 分组）
    for col in NUMERIC_MISSING_COLS:
        n_before_step = len(df)
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            rule_desc = f"使用全局中位数填补（中位数={median_val:.2f}）"
        else:
            rule_desc = "该列无缺失值，无需填补"
        logs.append({
            "处理步骤": f"数值缺失值填补: {col}",
            "处理规则": rule_desc,
            "处理前记录数": n_before_step,
            "处理后记录数": len(df),
            "影响记录数": int(missing_count)
        })
    
    # 3. 类别标准化
    for col, mapping in CATEGORY_MAPPINGS.items():
        for old_val, new_val in mapping.items():
            n_before_step = len(df)
            affected = (df[col] == old_val).sum()
            df[col] = df[col].replace(old_val, new_val)
            logs.append({
                "处理步骤": f"类别标准化: {col}",
                "处理规则": f"'{old_val}' → '{new_val}'",
                "处理前记录数": n_before_step,
                "处理后记录数": len(df),
                "影响记录数": int(affected)
            })
    
    # 4. 数据类型转换
    n_before_step = len(df)
    for col in ["Churn", "Complain"]:
        if col in df.columns:
            df[col] = df[col].astype(int)
    logs.append({
        "处理步骤": "数据类型转换",
        "处理规则": "将 Churn 和 Complain 转换为 int 类型，确保目标变量与二元标志类型一致",
        "处理前记录数": n_before_step,
        "处理后记录数": len(df),
        "影响记录数": len(df)
    })
    
    cleaned_df = df.reset_index(drop=True)
    cleaning_log = pd.DataFrame(logs)
    
    return cleaned_df, cleaning_log
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    # TODO：创建日志列表 logs
    # TODO：删除完全重复行，并记录日志
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    # TODO：将 Churn 和 Complain 转为整数类型
    # TODO：返回 cleaned_df 与 cleaning_log

### 任务 3：运行清洗函数并查看日志

In [17]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,删除所有列完全相同的重复记录，保留首次出现,5630,5630,0
1,数值缺失值填补: Tenure,使用全局中位数填补（中位数=9.00）,5630,5630,264
2,数值缺失值填补: WarehouseToHome,使用全局中位数填补（中位数=14.00）,5630,5630,251
3,数值缺失值填补: HourSpendOnApp,使用全局中位数填补（中位数=3.00）,5630,5630,255
4,数值缺失值填补: OrderAmountHikeFromlastYear,使用全局中位数填补（中位数=15.00）,5630,5630,265
5,数值缺失值填补: CouponUsed,使用全局中位数填补（中位数=1.00）,5630,5630,256
6,数值缺失值填补: OrderCount,使用全局中位数填补（中位数=2.00）,5630,5630,258
7,数值缺失值填补: DaySinceLastOrder,使用全局中位数填补（中位数=3.00）,5630,5630,307
8,类别标准化: PreferredLoginDevice,'Phone' → 'Mobile Phone',5630,5630,1231
9,类别标准化: PreferredPaymentMode,'COD' → 'Cash on Delivery',5630,5630,365


字段名,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [18]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = (series < lower) | (series > upper)
    outlier_values = series[outliers].tolist()

    return {
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "下限": round(lower, 2),
        "上限": round(upper, 2),
        "候选异常值数量": int(outliers.sum()),
        "异常值占比(%)": round(outliers.sum() / len(series) * 100, 2),
        "异常值示例": outlier_values[:5]
    }


# 1. 构建 TenureGroup：用户使用时长分层
# 业务分箱逻辑：新用户(0-6月)、短期(7-12月)、中期(13-24月)、长期(25月+)
tenure_bins = [0, 6, 12, 24, float('inf')]
tenure_labels = ["新用户(0-6月)", "短期用户(7-12月)", "中期用户(13-24月)", "长期用户(25月+)"]

cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    right=True,
    include_lowest=True
)

# 2. 构建 IsMobileLogin：是否主要使用移动端登录
# 清洗后 PreferredLoginDevice 只有 "Mobile Phone" 和 "Computer"
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

# 3. 生成候选异常值报告（仅记录，不删除）
outlier_fields = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = pd.DataFrame([
    {"字段名": col, **iqr_outlier_summary(cleaned_df[col])}
    for col in outlier_fields
])

# 展示结果
print("=== TenureGroup 分布 ===")
print(cleaned_df["TenureGroup"].value_counts().sort_index())

print("\n=== IsMobileLogin 分布 ===")
print(cleaned_df["IsMobileLogin"].value_counts())

print("\n=== 候选异常值报告（IQR 方法，仅记录不删除）===")
display(outlier_report)

# 详细边界说明
print("\n=== 异常值边界详细说明 ===")
for col in outlier_fields:
    stats = iqr_outlier_summary(cleaned_df[col])
    print(f"\n{col}:")
    print(f"  Q1={stats['Q1']}, Q3={stats['Q3']}, IQR={stats['IQR']}")
    print(f"  正常范围: [{stats['下限']}, {stats['上限']}]")
    print(f"  候选异常值数量: {stats['候选异常值数量']} ({stats['异常值占比(%)']}%)")
    print(f"  异常值示例: {stats['异常值示例']}")

=== TenureGroup 分布 ===
TenureGroup
新用户(0-6月)       2150
短期用户(7-12月)     1584
中期用户(13-24月)    1467
长期用户(25月+)       429
Name: count, dtype: int64

=== IsMobileLogin 分布 ===
IsMobileLogin
1    3996
0    1634
Name: count, dtype: int64

=== 候选异常值报告（IQR 方法，仅记录不删除）===


,字段名,Q1,Q3,IQR,下限,上限,候选异常值数量,异常值占比(%),异常值示例
0,WarehouseToHome,9.00,20.00,11.00,-7.50,36.50,2,0.04,"[126.0, 127.0]"
1,OrderCount,1.00,3.00,2.00,-2.00,6.00,703,12.49,"[15.0, 7.0, 15.0, 7.0, 7.0]"
2,CashbackAmount,145.77,196.39,50.62,69.84,272.33,438,7.78,"[295.45, 299.26, 290.33, 287.22, 299.99]"



=== 异常值边界详细说明 ===

WarehouseToHome:
  Q1=9.0, Q3=20.0, IQR=11.0
  正常范围: [-7.5, 36.5]
  候选异常值数量: 2 (0.04%)
  异常值示例: [126.0, 127.0]

OrderCount:
  Q1=1.0, Q3=3.0, IQR=2.0
  正常范围: [-2.0, 6.0]
  候选异常值数量: 703 (12.49%)
  异常值示例: [15.0, 7.0, 15.0, 7.0, 7.0]

CashbackAmount:
  Q1=145.77, Q3=196.39, IQR=50.62
  正常范围: [69.84, 272.33]
  候选异常值数量: 438 (7.78%)
  异常值示例: [295.45, 299.26, 290.33, 287.22, 299.99]


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [19]:
rules = [
    ("使用时长小于 0", cleaned_df["Tenure"] < 0),
    ("仓库距离小于 0", cleaned_df["WarehouseToHome"] < 0),
    ("订单数小于或等于 0", cleaned_df["OrderCount"] <= 0),
    ("返现金额小于 0", cleaned_df["CashbackAmount"] < 0),
]

business_rule_report = pd.DataFrame({
    "规则": [rule[0] for rule in rules],
    "不合规记录数": [int(rule[1].sum()) for rule in rules]
})

display(business_rule_report)

# 处理结论
print("\n" + "="*60)
print("处理结论")
print("="*60)

for _, row in business_rule_report.iterrows():
    rule = row["规则"]
    count = row["不合规记录数"]
    if count == 0:
        print(f"✅ {rule}：未发现不合规记录，数据质量良好，无需处理。")
    else:
        print(f"⚠️ {rule}：发现 {count} 条不合规记录，需进一步核查数据源或业务逻辑。")

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0



处理结论
✅ 使用时长小于 0：未发现不合规记录，数据质量良好，无需处理。
✅ 仓库距离小于 0：未发现不合规记录，数据质量良好，无需处理。
✅ 订单数小于或等于 0：未发现不合规记录，数据质量良好，无需处理。
✅ 返现金额小于 0：未发现不合规记录，数据质量良好，无需处理。


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [20]:
# 1) 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 2) 断言检查
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值缺失列仍有缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "Phone 未合并"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "COD 未合并"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "CC 未合并"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "新增字段缺失"
print("✅ 所有断言检查通过")

# 3) 导出交付文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

# 4) 输出异常值报告与业务规则报告
print("\n=== 候选异常值报告（仅记录，未删除）===")
display(outlier_report)

print("\n=== 业务规则检查报告 ===")
display(business_rule_report)

# 5) 清洗前后缺失值对比
print("\n=== 清洗前后缺失值对比 ===")
comparison = pd.DataFrame({
    "清洗前缺失": quality_before["缺失数量"],
    "清洗后缺失": quality_after["缺失数量"],
})
comparison["变化"] = comparison["清洗前缺失"] - comparison["清洗后缺失"]
print(comparison[comparison["变化"] > 0])

# 6) 交付文件路径
print("\n" + "=" * 60)
print("交付文件路径")
print("=" * 60)
for f in OUTPUT_DIR.iterdir():
    print(f"📁 {f.name}")

✅ 所有断言检查通过

=== 候选异常值报告（仅记录，未删除）===


,字段名,Q1,Q3,IQR,下限,上限,候选异常值数量,异常值占比(%),异常值示例
0,WarehouseToHome,9.00,20.00,11.00,-7.50,36.50,2,0.04,"[126.0, 127.0]"
1,OrderCount,1.00,3.00,2.00,-2.00,6.00,703,12.49,"[15.0, 7.0, 15.0, 7.0, 7.0]"
2,CashbackAmount,145.77,196.39,50.62,69.84,272.33,438,7.78,"[295.45, 299.26, 290.33, 287.22, 299.99]"



=== 业务规则检查报告 ===


,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0



=== 清洗前后缺失值对比 ===
                             清洗前缺失  清洗后缺失     变化
字段名                                             
CouponUsed                  256.00      0 256.00
DaySinceLastOrder           307.00      0 307.00
HourSpendOnApp              255.00      0 255.00
OrderAmountHikeFromlastYear 265.00      0 265.00
OrderCount                  258.00      0 258.00
Tenure                      264.00      0 264.00
WarehouseToHome             251.00      0 251.00

交付文件路径
📁 business_rule_report.csv
📁 cleaning_log.csv
📁 data_quality_after.csv
📁 data_quality_before.csv
📁 ecommerce_customer_cleaned.csv
📁 outlier_report.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

发现的质量问题： 7个数值字段缺失率4.5%~5.5%；3个类别字段存在同义词不一致；3个字段存在候选异常值。
处理策略： 缺失值用总体中位数填补（避免Churn信息泄露）；类别不一致用映射表标准化；异常值IQR识别仅记录，不盲目删除。
可作为第五天输入的原因： 缺失已清零、类别已统一、类型正确，新增TenureGroup和IsMobileLogin两个派生特征，全过程有日志可追溯。
需业务确认： OrderCount和CashbackAmount的候选异常值是否真实合理；TenureGroup分箱边界是否符合业务定义。